In [1]:
! pip install langchain_community tiktoken langchain-openai langchainhub chromadb langchain

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 1.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.7/84.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 484.9/484.9 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 13.2 MB/s eta 0:00:

`(2) LangSmith`

https://docs.smith.langchain.com/

In [ ]:
import os
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_API_KEY'] = 'lsv2_pt_false_keylse_keyffalse_keyalse_key0a_ea3f7f591f'

`(3) API Keys`

In [ ]:
os.environ['GROQ_API_KEY'] = 'gsk_false_keyse_keyfYfalse_keydyb3Ffalse_keyxt83qSYFM8ZZulKaus'

In [3]:
import os

print("GROQ_API_KEY:", os.environ.get("GROQ_API_KEY")[:10], "...")
print("LANGCHAIN_API_KEY:", os.environ.get("LANGCHAIN_API_KEY")[:10], "...")
print("LANGCHAIN_TRACING_V2:", os.environ.get("LANGCHAIN_TRACING_V2"))


GROQ_API_KEY: gsk_CFUfea ...
LANGCHAIN_API_KEY: lsv2_pt_5d ...
LANGCHAIN_TRACING_V2: true


In [4]:
!pip install -U langchain-groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 2.2 MB/s eta 0:00:00


In [5]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

response = llm.invoke("Reply with the word: OK")
print(response.content)


OK


## Part 1: Overview

[RAG quickstart](https://python.langchain.com/docs/use_cases/question_answering/quickstart)

In [6]:
!pip install -U \
  langchain \
  langchain-community \
  langchain-core \
  langchain-text-splitters \
  langchain-groq \
  chromadb \
  sentence-transformers \
  beautifulsoup4


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 2.1 MB/s eta 0:00:00
  Attempting uninstall: beautifulsoup4
    Found existing installation: beautifulsoup4 4.13.5
    Uninstalling beautifulsoup4-4.13.5:
      Successfully uninstalled beautifulsoup4-4.13.5


In [7]:
# ===================== INSTALL (si nécessaire) =====================
# Décommente cette partie si les librairies ne sont pas encore installées
# !pip install -U langchain langchain-community langchain-core langchain-text-splitters \
#   langchain-groq chromadb sentence-transformers beautifulsoup4

# ===================== IMPORTS =====================
import bs4

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate

from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings

# ===================== INDEXING =====================
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)

docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
splits = text_splitter.split_documents(docs)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings
)

retriever = vectorstore.as_retriever()

# ===================== PROMPT =====================
prompt = ChatPromptTemplate.from_template("""
You are an assistant for question-answering tasks.
Use the following context to answer the question.
If you don't know the answer, say you don't know.

Context:
{context}

Question:
{question}
""")

# ===================== LLM =====================
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

# ===================== RAG CHAIN =====================
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# ===================== TEST =====================
rag_chain.invoke("What is Task Decomposition?")


/tmp/ipython-input-1177516673.py:37: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

"Task Decomposition is the process of breaking down a complex task into smaller, more manageable subtasks or steps. This is often necessary for planning and achieving a goal, as it allows an agent or system to focus on one task at a time and make progress towards the overall objective. In the context of Large Language Models (LLMs), task decomposition can be facilitated through techniques such as Chain of Thought (CoT) and Tree of Thoughts, which help to transform complex tasks into multiple manageable tasks and shed light on the model's thinking process."

## Part 2: Indexing

![Screenshot 2024-02-12 at 1.36.56 PM.png](attachment:d1c0f19e-1f5f-4fc6-a860-16337c1910fa.png)

In [8]:
# Documents
question = "What kinds of pets do I like?"
document = "My favorite pet is a cat."

[Count tokens](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb) considering [~4 char / token](https://help.openai.com/en/articles/4936856-what-are-tokens-and-how-to-count-them)

In [9]:
import tiktoken

def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

num_tokens_from_string(question, "cl100k_base")

8

[Text embedding models](https://python.langchain.com/docs/integrations/text_embedding/openai)

In [10]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embd = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)
#we'll use this time this embedding model instead of all-MiniLM-L6-v2
query_result = embd.embed_query(question)
document_result = embd.embed_documents([document])[0]

print(len(query_result))


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

768


[Cosine similarity](https://platform.openai.com/docs/guides/embeddings/frequently-asked-questions) is recommended (1 indicates identical) for OpenAI embeddings.

In [11]:
import numpy as np

def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    return dot_product / (norm_vec1 * norm_vec2)

similarity = cosine_similarity(query_result, document_result)
print("Cosine Similarity:", similarity)

Cosine Similarity: 0.6988199944891462


[Document Loaders](https://python.langchain.com/docs/integrations/document_loaders/)

In [12]:
#### INDEXING ####

# Load blog
import bs4
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
blog_docs = loader.load()

[Splitter](https://python.langchain.com/docs/modules/data_connection/document_transformers/recursive_text_splitter)

> This text splitter is the recommended one for generic text. It is parameterized by a list of characters. It tries to split on them in order until the chunks are small enough. The default list is ["\n\n", "\n", " ", ""]. This has the effect of trying to keep all paragraphs (and then sentences, and then words) together as long as possible, as those would generically seem to be the strongest semantically related pieces of text.

In [15]:
# Split
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300,
    chunk_overlap=50)

# Make splits
splits = text_splitter.split_documents(blog_docs)

[Vectorstores](https://python.langchain.com/docs/integrations/vectorstores/)

In [18]:
# Index
from langchain_community.vectorstores import Chroma
vectorstore_2 = Chroma.from_documents(documents=splits,
                                    embedding=embd,collection_name="bge_collection")

retriever_2 = vectorstore_2.as_retriever()

## Part 3: Retrieval

In [27]:
# Index
retriever_2 = vectorstore_2.as_retriever(search_kwargs={"k": 4})  #here k is adjustable

In [28]:
docs = retriever_2.invoke("What is Task Decomposition?")

In [29]:
len(docs)

4

## Part 4: Generation

![Screenshot 2024-02-12 at 1.37.38 PM.png](attachment:f9b0e284-58e4-4d33-9594-2dad351c569a.png)

In [31]:
from langchain_core.prompts import ChatPromptTemplate

# Prompt
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)
print(prompt)


input_variables=['context', 'question'] input_types={} partial_variables={} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\n{context}\n\nQuestion: {question}\n'), additional_kwargs={})]


In [32]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

In [33]:
# Chain
chain = prompt | llm

In [39]:
result = chain.invoke({"context": docs, "question": "What is Task Decomposition?"})

# Pretty print the text content
print("\n=== Answer ===\n")
print(result.content)



=== Answer ===

Task Decomposition is the process of breaking down a complicated task into smaller and simpler steps. This is achieved by instructing the model to "think step by step" or by using techniques such as Chain of Thought (CoT) or Tree of Thoughts (Yao et al. 2023). It transforms big tasks into multiple manageable tasks and sheds light on the model's thinking process. Task decomposition can be done through various methods, including:

1. Using simple prompting like "Steps for XYZ.\\n1."
2. Using task-specific instructions, such as "Write a story outline" for writing a novel.
3. With human inputs.


In [38]:
print(type(result))

<class 'langchain_core.messages.ai.AIMessage'>


[RAG chains](https://python.langchain.com/docs/expression_language/get_started#rag-search-example)

In [52]:
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning, module="jupyter_client.session")

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {"context": retriever_2, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

result =rag_chain.invoke("What is Task Decomposition?")
print("\n=== Answer ===\n")
print(result)



=== Answer ===

Task Decomposition is the process of breaking down a complicated task into smaller and simpler steps. This is achieved by instructing the model to "think step by step" or by using techniques such as Chain of Thought (CoT) or Tree of Thoughts (Yao et al. 2023). It transforms big tasks into multiple manageable tasks and sheds light on the model's thinking process. Task decomposition can be done through various methods, including:

1. Using simple prompting like "Steps for XYZ.\\n1."
2. Using task-specific instructions, such as "Write a story outline" for writing a novel.
3. With human inputs.
